# KNN Sınıflandırması — Digits Veri Seti

Bu notebook'ta K-En Yakın Komşu (KNN) algoritmasını el yazısı rakam tanıma (Digits) veri seti üzerinde uyguluyoruz.

**Hedef:** 8x8 piksel gri tonlamalı görüntülerden rakamı tahmin etmek (0-9 arası 10 sınıf)

## 1. Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA

## 2. Veri Seti

In [ ]:
digits = load_digits()

print("Görüntü boyutu  :", digits.images[0].shape)
print("Özellik sayısı  :", digits.data.shape[1], "(8x8 piksel)") 
print("Örnek sayısı    :", digits.data.shape[0])
print("Sınıf sayısı    :", len(digits.target_names))
print("Sınıflar        :", digits.target_names)

In [ ]:
# Her rakamdan birer örnek göster
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit, ax in zip(range(10), axes.flatten()):
    idx = np.where(digits.target == digit)[0][0]
    ax.imshow(digits.images[idx], cmap='gray_r', interpolation='nearest')
    ax.set_title(f'Rakam: {digit}', fontsize=11)
    ax.axis('off')

plt.suptitle('Digits Veri Seti — Örnek Görüntüler (8x8 piksel)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('digits_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sınıf dağılımı
unique, counts = np.unique(digits.target, return_counts=True)
plt.figure(figsize=(9, 4))
bars = plt.bar(unique, counts, color=plt.cm.tab10(np.linspace(0, 1, 10)), edgecolor='white', linewidth=0.5)
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(count), ha='center', va='bottom', fontsize=10)
plt.xlabel('Rakam')
plt.ylabel('Örnek Sayısı')
plt.title('Sınıf Dağılımı')
plt.xticks(unique)
plt.tight_layout()
plt.savefig('digits_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Piksel Yoğunluğu Görselleştirmesi

In [ ]:
# Her rakam için ortalama görüntü
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit, ax in zip(range(10), axes.flatten()):
    avg_img = digits.images[digits.target == digit].mean(axis=0)
    im = ax.imshow(avg_img, cmap='hot', interpolation='nearest')
    ax.set_title(f'Ort. Rakam {digit}', fontsize=10)
    ax.axis('off')

plt.suptitle('Her Rakamın Ortalama Piksel Yoğunluğu', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('digits_avg_pixels.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Ön İşleme

In [ ]:
X = digits.data
y = digits.target

# Min-Max Normalizasyon
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# Train-Test Split (%70 eğitim, %30 test)
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Eğitim seti   : {X_train.shape[0]} örnek")
print(f"Test seti     : {X_test.shape[0]} örnek")
print(f"Özellik sayısı: {X_train.shape[1]} (piksel)")

## 5. En İyi k Değerini Bulma

In [ ]:
k_values = range(1, 16)
train_scores = []
test_scores = []
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))
    cv = cross_val_score(knn, X_normalized, y, cv=5)
    cv_scores.append(cv.mean())

best_k = list(k_values)[np.argmax(cv_scores)]
print(f"En iyi k değeri (5-fold CV): {best_k}")
print(f"CV doğruluğu               : {max(cv_scores):.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(k_values, train_scores, 'o-', label='Eğitim', color='#3498db')
plt.plot(k_values, test_scores, 's-', label='Test', color='#e74c3c')
plt.plot(k_values, cv_scores, '^--', label='5-Fold CV', color='#2ecc71')
plt.axvline(x=best_k, color='gray', linestyle='--', alpha=0.7, label=f'En iyi k={best_k}')
plt.xlabel('k (Komşu Sayısı)')
plt.ylabel('Doğruluk')
plt.title('k Değerine Göre KNN Performansı — Digits')
plt.legend()
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('digits_k_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Eğitimi ve Değerlendirme

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test Doğruluğu : {acc:.4f} ({acc*100:.2f}%)")
print(f"Doğru tahmin   : {np.sum(y_pred == y_test)} / {len(y_test)}")

In [ ]:
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=0.4)
plt.xlabel('Tahmin Edilen Rakam')
plt.ylabel('Gerçek Rakam')
plt.title(f'Confusion Matrix — Digits (k={best_k})', fontsize=13)
plt.tight_layout()
plt.savefig('digits_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. PCA ile 2D Görselleştirme

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_normalized)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y,
                      cmap='tab10', alpha=0.6, s=20, edgecolors='none')
plt.colorbar(scatter, ticks=range(10), label='Rakam')
plt.xlabel(f'PCA Bileşen 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PCA Bileşen 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Digits Veri Seti — PCA ile 2D Görselleştirme')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('digits_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Açıklanan toplam varyans: {sum(pca.explained_variance_ratio_)*100:.1f}%")

## 8. Yanlış Tahmin Edilen Örnekler

In [ ]:
wrong_idx = np.where(y_pred != y_test)[0]
print(f"Toplam hatalı tahmin: {len(wrong_idx)}")

show = min(10, len(wrong_idx))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in zip(wrong_idx[:show], axes.flatten()):
    img = X_test[i].reshape(8, 8)
    ax.imshow(img, cmap='gray_r')
    ax.set_title(f'Gerçek: {y_test[i]}\nTahmin: {y_pred[i]}', fontsize=9, color='red')
    ax.axis('off')

plt.suptitle('Yanlış Tahmin Edilen Örnekler', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('digits_wrong_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Sonuç

| Metrik | Değer |
|---|---|
| Algoritma | K-En Yakın Komşu (KNN) |
| Veri Seti | Digits (1797 örnek, 64 özellik, 10 sınıf) |
| En iyi k | CV ile belirlendi |
| Normalizasyon | Min-Max Scaler |
| Test Doğruluğu | Notebook çalıştırıldığında görünür |